# Data prep — prospective scenarios

Extends `01_dataPrep.ipynb` with a **prospective** dimension: recomputes `unit_burdens.csv`
against `premise`-generated future-background ecoinvent databases (electricity mix, cement,
steel efficiency updated to match an IAM scenario/year), instead of just the present-day
ecoinvent-3.12-cutoff snapshot.

**Run `01_dataPrep.ipynb` first** — this notebook assumes the baseline ecoinvent database and
`unit_burdens.csv` already exist, and only *adds* scenario rows to that same CSV (tagged by a
new `scenario` column; existing rows are back-filled as `"baseline"`).

**Scope**: one IAM model, several pathways, one fixed year (see `SCENARIOS` below) — not a
full model × pathway × year matrix. `benefits_constants.csv` and `eol_constants.csv` are
untouched: carbon content, rotation period, and CFF allocation factors don't depend on the
energy-transition pathway.

**Important caveat to keep in mind when reading results**: only the parts of a material's or
process's supply chain that route through premise-updated sectors will actually shift between
scenarios (e.g. anything relying on electricity, transport, cement, or steel upstream) — purely
biological/agricultural burdens (e.g. raw biomass cultivation) may barely move. That's expected,
not a bug.


## Notebook setup

#### Version pinning and environment variables

In [1]:
import os
import importlib.metadata

import bw2data as bd
import bw2io as bi
import bw2calc as bc
from dotenv import load_dotenv
from premise import NewDatabase

load_dotenv()  # reads .env in the repo root, if present

# ── Version pinning ───────────────────────────────────────────────────────────
for pkg in ["bw2data", "bw2io", "bw2calc", "premise"]:
    print(f"  {pkg}: {importlib.metadata.version(pkg)}")
print()


14:15:27-0400 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


/Users/tsuitpy/miniconda3/envs/brightway_env/lib/python3.11/site-packages/bw2calc/__init__.py:52: UserWarning: 
It seems like you have an AMD/INTEL x64 architecture, but haven't installed pypardiso:

    https://pypi.org/project/pypardiso/

Installing it could give you much faster calculations.

  warnings.warn(PYPARDISO_WARNING)


  bw2data: 4.4.2
  bw2io: 0.9.6
  bw2calc: 2.0.1
  premise: 2.4.9.2



#### Project and database constants

Same constants as `01_dataPrep.ipynb` — must match, since we're extending its output.

In [2]:
PROJECT_NAME     = "biobased_construction_impact_calculator"
EI_VERSION       = "3.12"
EI_MODEL         = "cutoff"
EI_DB_NAME       = f"ecoinvent-{EI_VERSION}-{EI_MODEL}"       # "ecoinvent-3.12-cutoff"
BIOSPHERE_NAME   = f"ecoinvent-{EI_VERSION}-biosphere"        # "ecoinvent-3.12-biosphere"

METHOD_NAMESPACE = f"ecoinvent-{EI_VERSION}"                  # "ecoinvent-3.12"
METHOD_FAMILY    = "EF v3.1"

OUTPUT_DIR  = "../../data/processed"  # adjust to match your repo structure
OUTPUT_PATH = f"{OUTPUT_DIR}/unit_burdens.csv"

bd.projects.set_current(PROJECT_NAME)
print(f"Active project: {bd.projects.current}")

# ── This notebook only adds to what 01_dataPrep.ipynb already built ──────────
missing = [name for name in (EI_DB_NAME, BIOSPHERE_NAME) if name not in bd.databases]
if missing:
    raise RuntimeError(
        f"Missing Brightway database(s) {missing} — run 01_dataPrep.ipynb first."
    )
if not os.path.exists(OUTPUT_PATH):
    raise RuntimeError(f"{OUTPUT_PATH} not found — run 01_dataPrep.ipynb first.")

print(f"Baseline database:  {EI_DB_NAME}")
print(f"Baseline burdens:   {OUTPUT_PATH}")


Active project: biobased_construction_impact_calculator
Baseline database:  ecoinvent-3.12-cutoff
Baseline burdens:   ../../data/processed/unit_burdens.csv


#### `PREMISE_KEY`

Decryption key for premise's IAM data files. Set it in `.env` (see `.env.example`) or export it directly.

In [3]:
PREMISE_KEY = os.environ.get("PREMISE_KEY")
if not PREMISE_KEY:
    raise RuntimeError("Set PREMISE_KEY (in .env or the environment) before running this notebook.")


## 1. Generate scenario databases with premise

One IAM model, several pathways, one fixed year — edit `SCENARIOS` to change the pathway list
or year. Each entry becomes its own Brightway database via `premise.NewDatabase`, following
the pattern in `references/ref_notebooks/01_quickstart_brightway.ipynb`.

`SECTORS_TO_UPDATE` controls which parts of the background premise rewrites. It's set to `None`
here — a full `ndb.update()` across **all** default sectors, not restricted to just the ones
this model's BOM directly touches (`electricity`, `cement`, `steel`). This is a deliberate
choice, made even though it costs noticeably more runtime: the resulting scenario databases are
much larger/more heavily restructured than the baseline ecoinvent database, so each unit-burden
lookup in section 2 takes longer (each `bc.LCA(...).lci()` call has to build and factorize a
bigger technosphere matrix). That's why `extract_unit_burdens_for_scenario()` saves its progress
to `unit_burdens.csv` after every row instead of only once at the end — a crash partway through
a long run shouldn't cost you the activities already resolved. Pass a list (e.g.
`["electricity", "cement", "steel"]`) instead of `None` if you want the narrower, faster update.

In [4]:
SCENARIOS = [
    {"model": "image", "pathway": "SSP2-M", "year": 2050},
    {"model": "image", "pathway": "SSP2-L", "year": 2050},
]

SECTORS_TO_UPDATE = None  # deliberately updating all sectors, not just electricity/cement/steel


def scenario_label(s):
    """Human-readable id stored in unit_burdens.csv's `scenario` column."""
    return f"{s['model']}_{s['pathway']}_{s['year']}"


def scenario_db_name(s):
    """Brightway database name premise writes this scenario to."""
    return f"premise-{s['model']}-{s['pathway'].lower()}-{s['year']}"


for s in SCENARIOS:
    print(f"  {scenario_label(s):35s} -> {scenario_db_name(s)}")

  image_SSP2-M_2050                   -> premise-image-ssp2-m-2050
  image_SSP2-L_2050                   -> premise-image-ssp2-l-2050


#### Build and write (skips scenarios already in Brightway)

In [5]:
scenarios_to_build = [s for s in SCENARIOS if scenario_db_name(s) not in bd.databases]

print(f"{len(SCENARIOS) - len(scenarios_to_build)} scenario database(s) already exist — skipping.")
print(f"{len(scenarios_to_build)} to build: {[scenario_db_name(s) for s in scenarios_to_build]}")

if scenarios_to_build:
    ndb = NewDatabase(
        scenarios=[s.copy() for s in scenarios_to_build],
        source_db=EI_DB_NAME,
        source_version=EI_VERSION,
        source_type="brightway",
        system_model=EI_MODEL,
        biosphere_name=BIOSPHERE_NAME,
        key=PREMISE_KEY,
    )
    ndb.update(SECTORS_TO_UPDATE)

    output_names = [scenario_db_name(s) for s in scenarios_to_build]
    ndb.write_db_to_brightway(name=output_names)

    assert all(name in bd.databases for name in output_names)
    print(f"Wrote: {output_names}")


2 scenario database(s) already exist — skipping.
0 to build: []


## 2. Extract unit burdens per scenario

Same extraction logic as `01_dataPrep.ipynb`'s "Systematic burden extraction" section, run once
per scenario database instead of once against the baseline. Reproduced here (not imported) so
`01_dataPrep.ipynb` stays untouched and independently valid — see that notebook if this logic
needs to change, and update both.

#### EF v3.1 impact categories

LCIA methods are registered once per Brightway project against the shared biosphere, so this list is identical across the baseline and every premise scenario database — no need to recompute per scenario.

In [6]:
EF_METHODS = [
    m for m in bd.methods
    if m[0] == METHOD_NAMESPACE and m[1] == METHOD_FAMILY
]
print(f"Found {len(EF_METHODS)} EF v3.1 method tuples")

EF_UNITS = {
    'acidification':                                     'mol H+-eq',
    'climate change':                                     'kg CO2-eq',
    'climate change: biogenic':                           'kg CO2-eq',
    'climate change: fossil':                             'kg CO2-eq',
    'climate change: land use and land use change':       'kg CO2-eq',
    'ecotoxicity: freshwater':                            'CTUe',
    'ecotoxicity: freshwater, inorganics':                'CTUe',
    'ecotoxicity: freshwater, organics':                  'CTUe',
    'energy resources: non-renewable':                    'MJ',
    'eutrophication: freshwater':                         'kg P-eq',
    'eutrophication: marine':                             'kg N-eq',
    'eutrophication: terrestrial':                        'mol N-eq',
    'human toxicity: carcinogenic':                       'CTUh',
    'human toxicity: carcinogenic, inorganics':           'CTUh',
    'human toxicity: carcinogenic, organics':             'CTUh',
    'human toxicity: non-carcinogenic':                   'CTUh',
    'human toxicity: non-carcinogenic, inorganics':       'CTUh',
    'human toxicity: non-carcinogenic, organics':         'CTUh',
    'ionising radiation: human health':                   'kBq U235-eq',
    'land use':                                           'dimensionless (soil quality index)',
    'material resources: metals/minerals':                'kg Sb-eq',
    'ozone depletion':                                    'kg CFC-11-eq',
    'particulate matter formation':                       'disease incidence',
    'photochemical oxidant formation: human health':      'dimensionless (tropospheric ozone concentration increase)',
    'water use':                                           'm3 world-eq deprived',
}

missing_units = [m[2] for m in EF_METHODS if m[2] not in EF_UNITS]
if missing_units:
    print(f"⚠ {len(missing_units)} categories have no unit defined: {missing_units}")
else:
    print(f"All {len(EF_METHODS)} categories have a unit defined.")


Found 25 EF v3.1 method tuples
All 25 categories have a unit defined.


#### Helper functions

Same as `01_dataPrep.ipynb`, but taking `ei` as an explicit argument instead of closing over a module-level database — this notebook runs the same lookups against several different databases (one per scenario).

In [7]:
def find_ei(ei, name, location, ref_product=None):
    """Look up a single activity by exact name and location, in database `ei`."""
    results = [
        a for a in ei
        if a['name'] == name
        and a['location'] == location
        and (ref_product is None or a.get('reference product') == ref_product)
    ]
    if len(results) == 0:
        raise ValueError(f"Dataset not found in '{ei.name}': '{name}' | {location}")
    if len(results) > 1:
        print(f"  Warning: multiple matches for '{name}' | {location} in '{ei.name}' — using first")
    return results[0]


def search_ei(ei, keyword, max_results=20):
    """Broad keyword search — used to verify dataset names still resolve after premise's update."""
    keyword_lower = keyword.lower()
    results = [a for a in ei if keyword_lower in a['name'].lower()]
    print(f"Found {len(results)} dataset(s) matching '{keyword}' in '{ei.name}':")
    for a in results[:max_results]:
        print(f"  name:     {a['name']}")
        print(f"  location: {a['location']}")
        print(f"  ref prod: {a.get('reference product', '—')}")
        print()
    if len(results) > max_results:
        print(f"  ... and {len(results) - max_results} more, not shown")


def get_kg_conversion_factor(act):
    """kg per 1 reference unit — 1.0 for kg-based activities, else derived from 'wet mass'."""
    if (act.get("unit") or "").lower() == "kilogram":
        return 1.0
    for exc in act.production():
        wet_mass = exc.get("properties", {}).get("wet mass", {}).get("amount")
        if wet_mass:
            return wet_mass
    return None


def get_locations_from_market_group(ei, market_group_name, market_group_location, constituent_name):
    """Ecoinvent's regional market groups aggregate country-level markets as technosphere
    inputs. Return the location codes of those inputs, rather than hardcoding a list.

    Must be run against the baseline `ei` (ecoinvent-3.12-cutoff), not a premise scenario
    database — premise's electricity-sector update rewires this market group to route through
    its own IAM macro-regions (e.g. RER -> WEU) instead of individual countries, so this
    technosphere walk finds nothing there. The underlying country-level activities (e.g.
    "market for electricity, low voltage" | DE) still exist by name in every scenario
    database, just no longer reachable this way — see `EUROPEAN_LOCATIONS` below."""
    market_group_act = find_ei(ei, market_group_name, market_group_location)
    return sorted({
        exc.input["location"]
        for exc in market_group_act.technosphere()
        if exc.input.get("name") == constituent_name
    })

#### Load unit burden data sources

Same Google Sheet as `01_dataPrep.ipynb` — the set of materials/processes in scope doesn't change between scenarios, only their scores.

In [8]:
import pandas as pd
import re

SHEET_ID = "1Vx1XDlohZulOEaFFgliiqx3JaZp4rxqQ-LC8B0_qGW4"
GID = "1997851380"  # unit_burdens_dataSources tab

sheet_url = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv&gid={GID}"
df_sources = pd.read_csv(sheet_url, na_values=["n/a"])

df_sources = df_sources[df_sources["unit"].notna()].copy()

for col in ["activity_name", "unit", "material_source", "ecoinventDataset_name",
            "geographicalCoverage", "referenceProduct", "alternative_data_source"]:
    df_sources[col] = df_sources[col].astype("string").str.strip()

def normalize_unit(raw):
    return re.sub(r"^1\s*", "", str(raw)).strip()

df_sources["material_unit"] = df_sources["unit"].apply(normalize_unit)

print(f"Loaded {len(df_sources)} activities from '{sheet_url.split('gid=')[0]}...'")


Loaded 32 activities from 'https://docs.google.com/spreadsheets/d/1Vx1XDlohZulOEaFFgliiqx3JaZp4rxqQ-LC8B0_qGW4/export?format=csv&...'


#### Pea protein binder — precomputed in `biopol_lca`

Scenario-invariant: this is a separate Brightway project untouched by premise, so it's computed once and reused for every scenario, same as the baseline.

In [9]:
CATEGORY_RENAME_MAP = {"photochemical ozone formation": "photochemical oxidant formation: human health"}

bd.projects.set_current("biopol_lca")
binder_act = bd.get_node(name="pea protein binder production", database="lca_database_3DPrintedBiopol")

biopol_methods = [m for m in bd.methods if m[0] == "EF v3.1"]
binder_scores = {}
lca = bc.LCA({binder_act: 1}, biopol_methods[0])
lca.lci()
lca.lcia()
binder_scores[CATEGORY_RENAME_MAP.get(biopol_methods[0][1], biopol_methods[0][1])] = lca.score
for method in biopol_methods[1:]:
    lca.switch_method(method)
    lca.lcia()
    binder_scores[CATEGORY_RENAME_MAP.get(method[1], method[1])] = lca.score

bd.projects.set_current(PROJECT_NAME)  # switch back before continuing

print(f"Computed {len(binder_scores)} category scores for the pea protein binder")

Computed 25 category scores for the pea protein binder


#### Migrate `unit_burdens.csv` to add a `scenario` column

One-time, idempotent: existing rows (from `01_dataPrep.ipynb`) are back-filled as `"baseline"` if the column doesn't exist yet.

In [10]:
df_check = pd.read_csv(OUTPUT_PATH)
if "scenario" not in df_check.columns:
    df_check.insert(0, "scenario", "baseline")
    df_check.to_csv(OUTPUT_PATH, index=False)
    print(f"Migrated {OUTPUT_PATH}: added 'scenario' column, back-filled {len(df_check)} rows as 'baseline'")
else:
    print(f"{OUTPUT_PATH} already has a 'scenario' column — no migration needed")


../../data/processed/unit_burdens.csv already has a 'scenario' column — no migration needed


#### European location codes — resolved once, from the baseline database

Used for any sheet row whose `geographicalCoverage` is `"lookup: all european countries"`
(`energy use`, `avoided burden - incineration, electricity`). Resolved once against the
baseline `ecoinvent-3.12-cutoff` database and reused for every scenario — **not**
re-resolved per scenario database. Premise's electricity-sector update rewires
`"market group for electricity, low voltage"` (RER / Europe without Switzerland) to route
through its own IAM macro-region (e.g. `WEU`) instead of individual countries, so walking a
scenario database's version of this market group finds 0 countries. The individual
country-level markets (e.g. `"market for electricity, low voltage"` | `DE`) still exist by
name in every scenario database — they're just no longer reachable via that technosphere
link — so resolving the country list from the untouched baseline and looking each one up
directly by name still works fine.

In [11]:
ei_baseline = bd.Database(EI_DB_NAME)

EUROPEAN_LOCATIONS = get_locations_from_market_group(
    ei_baseline, "market group for electricity, low voltage", "RER", "market for electricity, low voltage",
) + get_locations_from_market_group(
    ei_baseline, "market group for electricity, low voltage", "Europe without Switzerland", "market for electricity, low voltage",
)

print(f"Found {len(EUROPEAN_LOCATIONS)} European country markets (resolved once, from '{EI_DB_NAME}')")

Found 40 European country markets (resolved once, from 'ecoinvent-3.12-cutoff')


#### Systematic burden extraction per scenario (with caching)

Same branching logic as `01_dataPrep.ipynb`:
- if `ecoinventDataset_name` is set:
    - if `geographicalCoverage == "lookup: all european countries"` → score for every country
    - else → single lookup at the given location
- elif `alternative_data_source == "road transport 50 km"` → use this scenario's transport activity, scaled by `tkm_per_kg`
- elif `alternative_data_source == "biopol_lca_database"` → pull from `binder_scores` (scenario-invariant)

Caching is now scoped by `(scenario, material_name, material_source)` instead of just
`(material_name, material_source)` — re-running this notebook only fills in whatever is
missing for each scenario, same incremental-safety behaviour as `01_dataPrep.ipynb`.

In [12]:
TRANSPORT_DISTANCE_KM = 50  # generic collection distance assumption, same as 01_dataPrep.ipynb
FORCE_REFRESH = []          # e.g. ["Hard wood"] to force re-download for specific materials
LOOKUP_SECONDS_ESTIMATE = 20

EXPECTED_N_CATEGORIES = len({m[2] for m in EF_METHODS})
KEY_COLS = ["scenario", "material_name", "material_source", "location", "impact_category"]
OUTPUT_COLS = [
    "scenario", "material_name", "material_unit", "location",
    "impact_category", "impact_category_unit", "score",
    "lca_database", "lca_method", "material_source",
]


def extract_unit_burdens_for_scenario(ei, label):
    """Runs the 01_dataPrep.ipynb extraction against `ei` (a premise scenario database),
    tags every row with `label`, and appends the result into unit_burdens.csv.

    Saves after every df_sources row is resolved (not just once at the end) — a run against
    a premise scenario can take much longer than the baseline, so losing all progress to one
    bad row partway through would be expensive."""
    print(f"\n{'='*80}\nExtracting unit burdens for scenario: {label}  (database: {ei.name})\n{'='*80}")

    # ── Resolve this scenario's transport reference activity ──────────────────────
    transport_row = df_sources[df_sources["activity_name"] == "Road transport"].iloc[0]
    transport_act = find_ei(
        ei, transport_row["ecoinventDataset_name"], transport_row["geographicalCoverage"],
        transport_row["referenceProduct"] if pd.notna(transport_row["referenceProduct"]) else None,
    )
    tkm_per_kg = TRANSPORT_DISTANCE_KM * 0.001

    # ── Cache state (disk), scoped to this scenario ──────────────────────────────
    def load_cache_state():
        df_raw = pd.read_csv(OUTPUT_PATH)
        df_scenario = df_raw[df_raw["scenario"] == label]
        category_counts = (
            df_scenario.groupby(["material_name", df_scenario["material_source"].fillna("")])
            ["impact_category"].nunique()
        )
        complete = set(category_counts[category_counts >= EXPECTED_N_CATEGORIES].index)
        incomplete = set(category_counts[category_counts < EXPECTED_N_CATEGORIES].index)
        stale = incomplete | {k for k in complete if k[0] in FORCE_REFRESH}

        is_stale_row = (df_raw["scenario"] == label) & df_raw.apply(
            lambda r: (r["material_name"], r["material_source"] if pd.notna(r["material_source"]) else "") in stale,
            axis=1,
        )
        df_kept = df_raw[~is_stale_row].copy()

        print(f"  {len(complete)} activities already complete for '{label}'")
        if incomplete:
            print(f"  re-downloading {len(incomplete)} incomplete: {sorted(k[0] for k in incomplete)}")
        return df_kept, complete - set(FORCE_REFRESH)

    ei_score_cache = {}

    def get_ei_scores(ecoinvent_name, location, ref_product, material_unit):
        """Builds ONE bc.LCA per activity and reuses it across all EF v3.1 methods via
        switch_method() — solving the technosphere system once instead of once per method.
        On premise's much bigger scenario databases, a fresh bc.LCA + lci() per method (the
        old approach) costs ~8s each; switch_method() + lcia() costs ~0.003s each — the
        difference between ~3 min and ~8s per activity."""
        key = (ecoinvent_name, location, ref_product, material_unit)
        if key in ei_score_cache:
            return ei_score_cache[key]
        try:
            act = find_ei(ei, ecoinvent_name, location, ref_product)
        except ValueError:
            ei_score_cache[key] = None
            return None
        kg_factor = get_kg_conversion_factor(act) if material_unit == "kg" else None
        if material_unit == "kg" and kg_factor is None:
            print(f"⚠ '{ecoinvent_name}' | {location}: no kg-conversion available — skipped")
            ei_score_cache[key] = None
            return None

        scores = {}
        lca = bc.LCA({act: 1}, EF_METHODS[0])
        lca.lci()
        lca.lcia()
        scores[EF_METHODS[0][2]] = lca.score / kg_factor if kg_factor else lca.score
        for method in EF_METHODS[1:]:
            lca.switch_method(method)
            lca.lcia()
            scores[method[2]] = lca.score / kg_factor if kg_factor else lca.score
        ei_score_cache[key] = scores
        return scores

    def lookup_keys_for_row(row):
        if pd.isna(row["ecoinventDataset_name"]):
            return []
        ref_product = row["referenceProduct"] if pd.notna(row["referenceProduct"]) else None
        locations = (
            EUROPEAN_LOCATIONS if row["geographicalCoverage"].lower() == "lookup: all european countries"
            else [row["geographicalCoverage"]]
        )
        return [(row["ecoinventDataset_name"], loc, ref_product, row["material_unit"]) for loc in locations]

    def source_or_none(row):
        return row["material_source"] if pd.notna(row["material_source"]) else None

    def make_rows(activity_name, material_unit, material_source, location, scores, lca_database, lca_method):
        return [{
            "scenario": label,
            "material_name": activity_name, "material_unit": material_unit, "location": location,
            "impact_category": cat, "score": score,
            "lca_database": lca_database, "lca_method": lca_method, "material_source": material_source,
        } for cat, score in scores.items()]

    def resolve_ecoinvent_row(row):
        rows, errors = [], []
        for ecoinvent_name, location, ref_product, material_unit in lookup_keys_for_row(row):
            scores = get_ei_scores(ecoinvent_name, location, ref_product, material_unit)
            if scores is None:
                errors.append((f"{row['activity_name']} | {location}", "resolution/conversion failed"))
                continue
            rows += make_rows(row["activity_name"], material_unit, source_or_none(row),
                               location, scores, ei.name, EF_METHODS[0][1])
        return rows, errors

    def resolve_transport_row(row):
        scores = {}
        lca = bc.LCA({transport_act: 1}, EF_METHODS[0])
        lca.lci()
        lca.lcia()
        scores[EF_METHODS[0][2]] = lca.score * tkm_per_kg
        for method in EF_METHODS[1:]:
            lca.switch_method(method)
            lca.lcia()
            scores[method[2]] = lca.score * tkm_per_kg
        return make_rows(row["activity_name"], row["material_unit"], source_or_none(row),
                          row["geographicalCoverage"], scores, ei.name, EF_METHODS[0][1]), []

    def resolve_biopol_row(row):
        return make_rows(row["activity_name"], row["material_unit"], source_or_none(row),
                          row["geographicalCoverage"], binder_scores, "biopol_lca", "EF v3.1"), []

    def resolve_row(row):
        if pd.notna(row["ecoinventDataset_name"]):
            return resolve_ecoinvent_row(row)
        if row["alternative_data_source"] == "road transport 50 km":
            return resolve_transport_row(row)
        if row["alternative_data_source"] == "biopol_lca_database":
            return resolve_biopol_row(row)
        return [], [(f"{row['activity_name']} | {row['geographicalCoverage']}",
                     "No ecoinventDataset_name or recognized alternative_data_source")]

    # ── Persist a batch of new rows immediately, so a crash later doesn't lose earlier work ──
    def save_rows(df_accumulated, new_rows):
        if not new_rows:
            return df_accumulated
        df_new_rows = pd.DataFrame(new_rows)
        df_new_rows["impact_category_unit"] = df_new_rows["impact_category"].map(EF_UNITS)
        df_combined = pd.concat([df_accumulated, df_new_rows], ignore_index=True)
        df_combined = df_combined.drop_duplicates(subset=KEY_COLS, keep="last")[OUTPUT_COLS]
        df_combined.to_csv(OUTPUT_PATH, index=False)
        return df_combined

    # ── Run ───────────────────────────────────────────────────────────────────
    df_accumulated, cached_keys = load_cache_state()
    n_before = len(df_accumulated)

    resolution_errors, n_new_rows = [], 0
    for _, row in df_sources.iterrows():
        cache_key = (row["activity_name"], source_or_none(row) or "")
        if cache_key in cached_keys:
            continue
        geo = row["geographicalCoverage"] if pd.notna(row["geographicalCoverage"]) else None
        print(f"  Processing '{row['activity_name']}' ({source_or_none(row) or '—'}, {geo or '—'}) ...")
        new_rows, errors = resolve_row(row)
        resolution_errors += errors
        if new_rows:
            df_accumulated = save_rows(df_accumulated, new_rows)
            n_new_rows += len(new_rows)
            print(f"    saved {len(new_rows)} row(s) — {len(df_accumulated)} total rows in {OUTPUT_PATH} now")

    print(f"  Downloaded {n_new_rows} new rows for '{label}'")
    if resolution_errors:
        print(f"  ⚠ {len(resolution_errors)} unresolved:")
        for name, err in resolution_errors:
            print(f"    {name}: {err}")

    print(f"  Total rows in {OUTPUT_PATH}: {len(df_accumulated)} ({n_before} before this run)")
    return df_accumulated

#### Run extraction for every scenario

In [13]:
for s in SCENARIOS:
    ei_scenario = bd.Database(scenario_db_name(s))
    extract_unit_burdens_for_scenario(ei_scenario, scenario_label(s))

print("\nDone. unit_burdens.csv now contains 'baseline' plus:", [scenario_label(s) for s in SCENARIOS])



Extracting unit burdens for scenario: image_SSP2-M_2050  (database: premise-image-ssp2-m-2050)


  28 activities already complete for 'image_SSP2-M_2050'
  Processing 'Hard wood' (virgin, GLO) ...


⚠ 'market for sawnwood, hardwood, raw' | GLO: no kg-conversion available — skipped
  Processing 'Soft wood' (virgin, GLO) ...


⚠ 'market for sawnwood, softwood, raw' | GLO: no kg-conversion available — skipped
  Processing 'energy use' (—, lookup: all european countries) ...


    saved 1000 row(s) — 5150 total rows in ../../data/processed/unit_burdens.csv now
  Processing 'avoided burden - incineration, electricity' (—, lookup: all european countries) ...
    saved 1000 row(s) — 6150 total rows in ../../data/processed/unit_burdens.csv now
  Downloaded 2000 new rows for 'image_SSP2-M_2050'
  ⚠ 2 unresolved:
    Hard wood | GLO: resolution/conversion failed
    Soft wood | GLO: resolution/conversion failed
  Total rows in ../../data/processed/unit_burdens.csv: 6150 (4150 before this run)

Extracting unit burdens for scenario: image_SSP2-L_2050  (database: premise-image-ssp2-l-2050)


  28 activities already complete for 'image_SSP2-L_2050'
  Processing 'Hard wood' (virgin, GLO) ...


⚠ 'market for sawnwood, hardwood, raw' | GLO: no kg-conversion available — skipped
  Processing 'Soft wood' (virgin, GLO) ...


⚠ 'market for sawnwood, softwood, raw' | GLO: no kg-conversion available — skipped
  Processing 'energy use' (—, lookup: all european countries) ...


    saved 1000 row(s) — 7150 total rows in ../../data/processed/unit_burdens.csv now
  Processing 'avoided burden - incineration, electricity' (—, lookup: all european countries) ...
    saved 1000 row(s) — 8150 total rows in ../../data/processed/unit_burdens.csv now
  Downloaded 2000 new rows for 'image_SSP2-L_2050'
  ⚠ 2 unresolved:
    Hard wood | GLO: resolution/conversion failed
    Soft wood | GLO: resolution/conversion failed
  Total rows in ../../data/processed/unit_burdens.csv: 8150 (6150 before this run)

Done. unit_burdens.csv now contains 'baseline' plus: ['image_SSP2-M_2050', 'image_SSP2-L_2050']


## 3. Validation

Two quick checks worth running before trusting these numbers in `02_model.ipynb`:

1. **Activity-resolution spot check** — premise occasionally restructures market activities
   (renames, splits into new region-specific ones). If `find_ei()` silently fails to resolve
   something post-transformation, it shows up above as a "resolution/conversion failed" error
   or a `search_ei` mismatch — re-run `search_ei(ei_scenario, "electricity")` (or the relevant
   keyword) against one scenario database if anything looks wrong.
2. **Sanity-check direction** — for an electricity-heavy row (e.g. `"energy use"`), the
   `"climate change"` unit burden should be *lower* under the more mitigation-ambitious of the
   two IMAGE pathways (`SSP2-L` is the lower-forcing/more-ambitious one relative to `SSP2-M` —
   worth double-checking against IMAGE's own scenario documentation rather than taking that as
   given) than under `baseline`. A wrong sign usually means a stale cache entry or a scenario
   mix-up.

In [14]:
df_burdens_check = pd.read_csv(OUTPUT_PATH)

check = df_burdens_check[
    (df_burdens_check["material_name"] == "energy use")
    & (df_burdens_check["impact_category"] == "climate change")
]
check.pivot_table(index="location", columns="scenario", values="score")


scenario,baseline,image_SSP2-L_2050,image_SSP2-M_2050
location,,,
AL,0.311124,0.029175,0.056292
AT,0.275082,0.021992,0.037337
BA,0.951789,0.029175,0.056292
BE,0.202075,0.021992,0.037337
BG,0.574338,0.029175,0.056292
BY,0.635167,0.092663,0.624653
CH,0.031382,0.021992,0.037337
CZ,0.676482,0.029175,0.056292
DE,0.460572,0.021992,0.037337
